## Final Project Submission

Please fill out:
* Student name: **GROUP 6**
* Student pace: **part time** 
* Scheduled project review date/time: 
* Instructor name: **FIDELIS WANALWENGE**
* Blog post URL:


In [1]:
#Importing necessary libraries
import pandas as pd
import sqlite3

In [2]:
#Checking working directory
import os
print(os.listdir())

['.git', '.ipynb_checkpoints', 'CONTRIBUTING.md', 'index.ipynb', 'LICENSE.md', 'movie_data_erd.jpeg', 'README.md', 'student.ipynb', 'zippedData']


In [3]:
#Navigating into zippedData
print(os.listdir("zippedData"))

['bom.movie_gross.csv.gz', 'im.db', 'im.db.zip', 'rt.movie_info.tsv.gz', 'rt.reviews.tsv.gz', 'tmdb.movies.csv.gz', 'tn.movie_budgets.csv.gz']


Connecting to the Database Using sqlite3 + pandas

In [4]:
conn = sqlite3.connect('zippedData\im.db\im.db')

In [5]:
#Creating a DataFrame that contains our table names
df_tables = pd.read_sql("""SELECT name AS table_name
                             FROM sqlite_master 
                            WHERE type = 'table';""", conn)
df_tables

,table_name
0,movie_basics
1,directors
2,known_for
3,movie_akas
4,movie_ratings
5,persons
6,principals
7,writers


## Having a look at the movie_basics table

In [6]:
df_movie_basics = pd.read_sql("""SELECT *
                                  FROM movie_basics;""", conn)
df_movie_basics

,movie_id,primary_title,original_title,start_year,runtime_minutes,genres
0,tt0063540,Sunghursh,Sunghursh,2013,175.0,"Action,Crime,Drama"
1,tt0066787,One Day Before the Rainy Season,Ashad Ka Ek Din,2019,114.0,"Biography,Drama"
2,tt0069049,The Other Side of the Wind,The Other Side of the Wind,2018,122.0,Drama
3,tt0069204,Sabse Bada Sukh,Sabse Bada Sukh,2018,NaN,"Comedy,Drama"
4,tt0100275,The Wandering Soap Opera,La Telenovela Errante,2017,80.0,"Comedy,Drama,Fantasy"
...,...,...,...,...,...,...
146139,tt9916538,Kuambil Lagi Hatiku,Kuambil Lagi Hatiku,2019,123.0,Drama
146140,tt9916622,Rodolpho Teóphilo - O Legado de um Pioneiro,Rodolpho Teóphilo - O Legado de um Pioneiro,2015,NaN,Documentary
146141,tt9916706,Dankyavar Danka,Dankyavar Danka,2013,NaN,Comedy
146142,tt9916730,6 Gunn,6 Gunn,2017,116.0,None


In [7]:
df_movie_basics.describe()

,start_year,runtime_minutes
count,146144.000000,114405.000000
mean,2014.621798,86.187247
std,2.733583,166.360590
min,2010.000000,1.000000
25%,2012.000000,70.000000
50%,2015.000000,87.000000
75%,2017.000000,99.000000
max,2115.000000,51420.000000


In [8]:
df_movie_basics.columns

Index(['movie_id', 'primary_title', 'original_title', 'start_year',
       'runtime_minutes', 'genres'],
      dtype='object')

In [9]:
#Checking primary_title column
df_movie_basics['primary_title']

0                                           Sunghursh
1                     One Day Before the Rainy Season
2                          The Other Side of the Wind
3                                     Sabse Bada Sukh
4                            The Wandering Soap Opera
                             ...                     
146139                            Kuambil Lagi Hatiku
146140    Rodolpho Teóphilo - O Legado de um Pioneiro
146141                                Dankyavar Danka
146142                                         6 Gunn
146143                 Chico Albuquerque - Revelações
Name: primary_title, Length: 146144, dtype: object

In [10]:
#Checking for unique values
df_movie_basics.nunique()

movie_id           146144
primary_title      136071
original_title     137773
start_year             19
runtime_minutes       367
genres               1085
dtype: int64

In [11]:
df_movie_basics['genres'].unique()

array(['Action,Crime,Drama', 'Biography,Drama', 'Drama', ...,
       'Music,Musical,Reality-TV', 'Animation,Crime',
       'Adventure,History,War'], dtype=object)

In [12]:
#Sorting rows using ORDER BY
pd.read_sql("""SELECT start_year
                FROM movie_basics
                ORDER BY start_year ASC;""", conn)

,start_year
0,2010
1,2010
2,2010
3,2010
4,2010
...,...
146139,2024
146140,2025
146141,2026
146142,2027


In [13]:
#GROUP BY for grouping rows
pd.read_sql("""SELECT start_year, genres
                FROM movie_basics
                GROUP BY genres
                ORDER BY start_year ASC;""", conn)

,start_year,genres
0,2010,"Action,Adventure,Crime"
1,2010,"Action,Adventure,Family"
2,2010,"Action,Adventure,Mystery"
3,2010,"Action,Adventure,Thriller"
4,2010,"Action,Adventure,War"
...,...,...
1081,2020,"Romance,Thriller"
1082,2021,"Action,Adventure,Drama"
1083,2021,"Action,Adventure,Fantasy"
1084,2021,"Drama,History,Western"


In [14]:
df_movie_basics.isnull().sum()

movie_id               0
primary_title          0
original_title        21
start_year             0
runtime_minutes    31739
genres              5408
dtype: int64

In [15]:
df_movie_basics['runtime_minutes'].mean()

86.18724706088021

In [16]:
df_movie_basics['runtime_minutes'].mode()

0    90.0
dtype: float64

In [17]:
df_movie_basics['genres'].mode()

0    Documentary
dtype: object

## Filling Missing values

In [18]:
#filling genres with mode
df_movie_basics['genres'].fillna(df_movie_basics['genres'].mode()[0], inplace=True)

In [19]:
#Checking whether genres column was filled
df_movie_basics['genres'].isnull().sum()

0

In [20]:
#filling empty original_title with primary_title values
df_movie_basics['original_title'] = df_movie_basics['original_title'].where(
    df_movie_basics['original_title'].notna(),
    df_movie_basics['primary_title']
)

In [21]:
print(df_movie_basics['original_title'].isna().sum())

0


In [22]:
#Filling runtime_mins with mode
df_movie_basics['runtime_minutes'].fillna(df_movie_basics['runtime_minutes'].mode()[0], inplace=True)

## Having a look at movie_ratings table

In [23]:
df_movie_ratings = pd.read_sql("""SELECT *
                FROM movie_ratings;""", conn)

In [24]:
pd.read_sql("""SELECT *
                FROM movie_ratings
                WHERE averagerating > 7.0
                ORDER BY averagerating DESC;""", conn)

,movie_id,averagerating,numvotes
0,tt5390098,10.0,5
1,tt6295832,10.0,5
2,tt1770682,10.0,5
3,tt2632430,10.0,5
4,tt8730716,10.0,5
...,...,...,...
24640,tt8319662,7.1,8
24641,tt8574252,7.1,1526
24642,tt8954044,7.1,8
24643,tt9471952,7.1,338


In [25]:
pd.read_sql("""SELECT *
                FROM movie_ratings
                WHERE averagerating > 7.0
                GROUP BY numvotes
                HAVING numvotes > 100
                ORDER BY averagerating DESC;""", conn)

,movie_id,averagerating,numvotes
0,tt7131622,9.7,5600
1,tt6058226,9.6,2604
2,tt4131686,9.6,1339
3,tt9343826,9.6,808
4,tt9680166,9.6,624
...,...,...,...
2980,tt3232316,7.1,122
2981,tt8819596,7.1,121
2982,tt7142712,7.1,114
2983,tt3030714,7.1,106


In [26]:
df_tables

,table_name
0,movie_basics
1,directors
2,known_for
3,movie_akas
4,movie_ratings
5,persons
6,principals
7,writers


In [27]:
#known_for table
pd.read_sql("""SELECT *
                FROM known_for;""", conn)

,person_id,movie_id
0,nm0061671,tt0837562
1,nm0061671,tt2398241
2,nm0061671,tt0844471
3,nm0061671,tt0118553
4,nm0061865,tt0896534
...,...,...
1638255,nm9990690,tt9090932
1638256,nm9990690,tt8737130
1638257,nm9991320,tt8734436
1638258,nm9991320,tt9615610


In [28]:
#directors table
pd.read_sql("""SELECT *
                FROM directors;""", conn)

,movie_id,person_id
0,tt0285252,nm0899854
1,tt0462036,nm1940585
2,tt0835418,nm0151540
3,tt0835418,nm0151540
4,tt0878654,nm0089502
...,...,...
291169,tt8999974,nm10122357
291170,tt9001390,nm6711477
291171,tt9001494,nm10123242
291172,tt9001494,nm10123248


In [29]:
#Principals table
pd.read_sql("""SELECT *
                FROM principals;""", conn)

,movie_id,ordering,person_id,category,job,characters
0,tt0111414,1,nm0246005,actor,None,"[""The Man""]"
1,tt0111414,2,nm0398271,director,None,None
2,tt0111414,3,nm3739909,producer,producer,None
3,tt0323808,10,nm0059247,editor,None,None
4,tt0323808,1,nm3579312,actress,None,"[""Beth Boothby""]"
...,...,...,...,...,...,...
1028181,tt9692684,1,nm0186469,actor,None,"[""Ebenezer Scrooge""]"
1028182,tt9692684,2,nm4929530,self,None,"[""Herself"",""Regan""]"
1028183,tt9692684,3,nm10441594,director,None,None
1028184,tt9692684,4,nm6009913,writer,writer,None


In [30]:
#writers table
pd.read_sql("""SELECT *
                FROM writers;""", conn)

,movie_id,person_id
0,tt0285252,nm0899854
1,tt0438973,nm0175726
2,tt0438973,nm1802864
3,tt0462036,nm1940585
4,tt0835418,nm0310087
...,...,...
255868,tt8999892,nm10122246
255869,tt8999974,nm10122357
255870,tt9001390,nm6711477
255871,tt9004986,nm4993825


## Checking for missing values

In [31]:
df_movie_ratings.isnull().sum()

movie_id         0
averagerating    0
numvotes         0
dtype: int64

Awesome! No missing values in the movie_ratings table

# Having a look at the csv files

In [32]:
import csv
#Bom_Movie
bom_movie = pd.read_csv(r'zippedData\bom.movie_gross.csv.gz')
print(bom_movie)

                                            title      studio  domestic_gross  \
0                                     Toy Story 3          BV     415000000.0   
1                      Alice in Wonderland (2010)          BV     334200000.0   
2     Harry Potter and the Deathly Hallows Part 1          WB     296000000.0   
3                                       Inception          WB     292600000.0   
4                             Shrek Forever After        P/DW     238700000.0   
...                                           ...         ...             ...   
3382                                    The Quake       Magn.          6200.0   
3383                  Edward II (2018 re-release)          FM          4800.0   
3384                                     El Pacto        Sony          2500.0   
3385                                     The Swan  Synergetic          2400.0   
3386                            An Actor Prepares       Grav.          1700.0   

     foreign_gross  year  


In [33]:
bom_movie.columns

Index(['title', 'studio', 'domestic_gross', 'foreign_gross', 'year'], dtype='object')

In [34]:
bom_movie.info

<bound method DataFrame.info of                                             title      studio  domestic_gross  \
0                                     Toy Story 3          BV     415000000.0   
1                      Alice in Wonderland (2010)          BV     334200000.0   
2     Harry Potter and the Deathly Hallows Part 1          WB     296000000.0   
3                                       Inception          WB     292600000.0   
4                             Shrek Forever After        P/DW     238700000.0   
...                                           ...         ...             ...   
3382                                    The Quake       Magn.          6200.0   
3383                  Edward II (2018 re-release)          FM          4800.0   
3384                                     El Pacto        Sony          2500.0   
3385                                     The Swan  Synergetic          2400.0   
3386                            An Actor Prepares       Grav.          1700.0

In [35]:
bom_movie.isna().sum()

title                0
studio               5
domestic_gross      28
foreign_gross     1350
year                 0
dtype: int64

In [36]:
bom_movie['domestic_gross'].mode()

0    1100000.0
dtype: float64

In [37]:
#Filling domestic_gross with mode
bom_movie['domestic_gross'].fillna(bom_movie['domestic_gross'].mode()[0], inplace=True)

In [38]:
#Filling foreign_gross with mode
bom_movie['foreign_gross'].fillna(bom_movie['foreign_gross'].mode()[0], inplace=True)

In [39]:
#Filling studio with mode
bom_movie['studio'].fillna(bom_movie['studio'].mode()[0], inplace=True)

In [40]:
#tn.movie budgets
movie_budget = pd.read_csv(r'zippedData\tn.movie_budgets.csv.gz')

In [41]:
movie_budget

,id,release_date,movie,production_budget,domestic_gross,worldwide_gross
0,1,"Dec 18, 2009",Avatar,"$425,000,000","$760,507,625","$2,776,345,279"
1,2,"May 20, 2011",Pirates of the Caribbean: On Stranger Tides,"$410,600,000","$241,063,875","$1,045,663,875"
2,3,"Jun 7, 2019",Dark Phoenix,"$350,000,000","$42,762,350","$149,762,350"
3,4,"May 1, 2015",Avengers: Age of Ultron,"$330,600,000","$459,005,868","$1,403,013,963"
4,5,"Dec 15, 2017",Star Wars Ep. VIII: The Last Jedi,"$317,000,000","$620,181,382","$1,316,721,747"
...,...,...,...,...,...,...
5777,78,"Dec 31, 2018",Red 11,"$7,000",$0,$0
5778,79,"Apr 2, 1999",Following,"$6,000","$48,482","$240,495"
5779,80,"Jul 13, 2005",Return to the Land of Wonders,"$5,000","$1,338","$1,338"
5780,81,"Sep 29, 2015",A Plague So Pleasant,"$1,400",$0,$0


In [42]:
movie_budget.isna().sum()

id                   0
release_date         0
movie                0
production_budget    0
domestic_gross       0
worldwide_gross      0
dtype: int64

In [43]:
#tmdb.movies
tmdb_movies = pd.read_csv(r'zippedData\tmdb.movies.csv.gz')
tmdb_movies

,Unnamed: 0,genre_ids,id,original_language,original_title,popularity,release_date,title,vote_average,vote_count
0,0,"[12, 14, 10751]",12444,en,Harry Potter and the Deathly Hallows: Part 1,33.533,2010-11-19,Harry Potter and the Deathly Hallows: Part 1,7.7,10788
1,1,"[14, 12, 16, 10751]",10191,en,How to Train Your Dragon,28.734,2010-03-26,How to Train Your Dragon,7.7,7610
2,2,"[12, 28, 878]",10138,en,Iron Man 2,28.515,2010-05-07,Iron Man 2,6.8,12368
3,3,"[16, 35, 10751]",862,en,Toy Story,28.005,1995-11-22,Toy Story,7.9,10174
4,4,"[28, 878, 12]",27205,en,Inception,27.920,2010-07-16,Inception,8.3,22186
...,...,...,...,...,...,...,...,...,...,...
26512,26512,"[27, 18]",488143,en,Laboratory Conditions,0.600,2018-10-13,Laboratory Conditions,0.0,1
26513,26513,"[18, 53]",485975,en,_EXHIBIT_84xxx_,0.600,2018-05-01,_EXHIBIT_84xxx_,0.0,1
26514,26514,"[14, 28, 12]",381231,en,The Last One,0.600,2018-10-01,The Last One,0.0,1
26515,26515,"[10751, 12, 28]",366854,en,Trailer Made,0.600,2018-06-22,Trailer Made,0.0,1


In [44]:
tmdb_movies.isna().sum()

Unnamed: 0           0
genre_ids            0
id                   0
original_language    0
original_title       0
popularity           0
release_date         0
title                0
vote_average         0
vote_count           0
dtype: int64

In [45]:
# Convert 'release_date' to datetime in movie_budget
movie_budget['release_date'] = pd.to_datetime(movie_budget['release_date'])

# Convert 'release_date' to datetime in tmdb_movies
tmdb_movies['release_date'] = pd.to_datetime(tmdb_movies['release_date'])

In [47]:
merged_bom_budget = pd.merge(bom_movie, movie_budget, left_on=['title', 'year'], right_on=['movie', movie_budget['release_date'].dt.year], how='inner')
merged_data = pd.merge(merged_bom_budget, tmdb_movies, left_on=['title', 'release_date'], right_on=['title', 'release_date'], how='inner')
display(merged_data.head())
merged_data.info()

,title,studio,domestic_gross_x,foreign_gross,year,id_x,release_date,movie,production_budget,domestic_gross_y,worldwide_gross,Unnamed: 0,genre_ids,id_y,original_language,original_title,popularity,vote_average,vote_count
0,Inception,WB,292600000.0,535700000,2010,38,2010-07-16,Inception,"$160,000,000","$292,576,195","$835,524,642",4,"[28, 878, 12]",27205,en,Inception,27.920,8.3,22186
1,Iron Man 2,Par.,312400000.0,311500000,2010,15,2010-05-07,Iron Man 2,"$170,000,000","$312,433,331","$621,156,389",2,"[12, 28, 878]",10138,en,Iron Man 2,28.515,6.8,12368
2,Tangled,BV,200800000.0,391000000,2010,15,2010-11-24,Tangled,"$260,000,000","$200,821,936","$586,477,240",13,"[16, 10751]",38757,en,Tangled,21.511,7.5,6407
3,Despicable Me,Uni.,251500000.0,291600000,2010,50,2010-07-09,Despicable Me,"$69,000,000","$251,513,985","$543,464,573",8,"[16, 10751, 35]",20352,en,Despicable Me,23.673,7.2,10057
4,How to Train Your Dragon,P/DW,217600000.0,277300000,2010,30,2010-03-26,How to Train Your Dragon,"$165,000,000","$217,581,232","$494,870,992",1,"[14, 12, 16, 10751]",10191,en,How to Train Your Dragon,28.734,7.7,7610


<class 'pandas.core.frame.DataFrame'>
Int64Index: 1014 entries, 0 to 1013
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   title              1014 non-null   object        
 1   studio             1014 non-null   object        
 2   domestic_gross_x   1014 non-null   float64       
 3   foreign_gross      1014 non-null   object        
 4   year               1014 non-null   int64         
 5   id_x               1014 non-null   int64         
 6   release_date       1014 non-null   datetime64[ns]
 7   movie              1014 non-null   object        
 8   production_budget  1014 non-null   object        
 9   domestic_gross_y   1014 non-null   object        
 10  worldwide_gross    1014 non-null   object        
 11  Unnamed: 0         1014 non-null   int64         
 12  genre_ids          1014 non-null   object        
 13  id_y               1014 non-null   int64         
 14  original

In [48]:
merged_data

,title,studio,domestic_gross_x,foreign_gross,year,id_x,release_date,movie,production_budget,domestic_gross_y,worldwide_gross,Unnamed: 0,genre_ids,id_y,original_language,original_title,popularity,vote_average,vote_count
0,Inception,WB,292600000.0,535700000,2010,38,2010-07-16,Inception,"$160,000,000","$292,576,195","$835,524,642",4,"[28, 878, 12]",27205,en,Inception,27.920,8.3,22186
1,Iron Man 2,Par.,312400000.0,311500000,2010,15,2010-05-07,Iron Man 2,"$170,000,000","$312,433,331","$621,156,389",2,"[12, 28, 878]",10138,en,Iron Man 2,28.515,6.8,12368
2,Tangled,BV,200800000.0,391000000,2010,15,2010-11-24,Tangled,"$260,000,000","$200,821,936","$586,477,240",13,"[16, 10751]",38757,en,Tangled,21.511,7.5,6407
3,Despicable Me,Uni.,251500000.0,291600000,2010,50,2010-07-09,Despicable Me,"$69,000,000","$251,513,985","$543,464,573",8,"[16, 10751, 35]",20352,en,Despicable Me,23.673,7.2,10057
4,How to Train Your Dragon,P/DW,217600000.0,277300000,2010,30,2010-03-26,How to Train Your Dragon,"$165,000,000","$217,581,232","$494,870,992",1,"[14, 12, 16, 10751]",10191,en,How to Train Your Dragon,28.734,7.7,7610
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1009,Destroyer,Annapurna,1500000.0,4000000,2018,5,2018-12-25,Destroyer,"$9,000,000","$1,533,324","$3,681,096",23921,"[53, 80, 18, 28]",471507,en,Destroyer,17.815,5.9,176
1010,Gotti,VE,4300000.0,1200000,2018,64,2018-06-15,Gotti,"$10,000,000","$4,286,367","$6,089,100",24168,"[80, 18, 36, 53]",339103,en,Gotti,10.034,5.2,231
1011,Bilal: A New Breed of Hero,VE,491000.0,1700000,2018,100,2018-02-02,Bilal: A New Breed of Hero,"$30,000,000","$490,973","$648,599",25148,"[28, 12, 16]",332718,en,Bilal: A New Breed of Hero,2.707,6.8,54
1012,Lean on Pete,A24,1200000.0,1200000,2018,13,2018-04-06,Lean on Pete,"$8,000,000","$1,163,056","$2,455,027",20908,"[18, 12]",407890,en,Lean on Pete,9.307,6.9,133


In [49]:
merged_data.to_csv("merged_data.csv", index=False)

# Having a look at tsv files

In [50]:
#rt.movie_info.tsv.gz
rt_movie = pd.read_csv(r'zippedData\rt.movie_info.tsv.gz', sep='\t', compression='gzip', encoding='utf-8')
print(rt_movie.head())

   id                                           synopsis rating  \
0   1  This gritty, fast-paced, and innovative police...      R   
1   3  New York City, not-too-distant-future: Eric Pa...      R   
2   5  Illeana Douglas delivers a superb performance ...      R   
3   6  Michael Douglas runs afoul of a treacherous su...      R   
4   7                                                NaN     NR   

                                 genre          director  \
0  Action and Adventure|Classics|Drama  William Friedkin   
1    Drama|Science Fiction and Fantasy  David Cronenberg   
2    Drama|Musical and Performing Arts    Allison Anders   
3           Drama|Mystery and Suspense    Barry Levinson   
4                        Drama|Romance    Rodney Bennett   

                            writer  theater_date      dvd_date currency  \
0                   Ernest Tidyman   Oct 9, 1971  Sep 25, 2001      NaN   
1     David Cronenberg|Don DeLillo  Aug 17, 2012   Jan 1, 2013        $   
2          

In [51]:
rt_movie.isna().sum()

id                 0
synopsis          62
rating             3
genre              8
director         199
writer           449
theater_date     359
dvd_date         359
currency        1220
box_office      1220
runtime           30
studio          1066
dtype: int64

In [52]:
#rt.reviews.tsv.gz
rt_reviews = pd.read_csv(
    r'zippedData\rt.reviews.tsv.gz',
    sep='\t',
    compression='gzip',
    encoding='latin1'   # instead of utf-8
)

print(rt_reviews.head())


   id                                             review rating   fresh  \
0   3  A distinctly gallows take on contemporary fina...    3/5   fresh   
1   3  It's an allegory in search of a meaning that n...    NaN  rotten   
2   3  ... life lived in a bubble in financial dealin...    NaN   fresh   
3   3  Continuing along a line introduced in last yea...    NaN   fresh   
4   3             ... a perverse twist on neorealism...     NaN   fresh   

           critic  top_critic         publisher               date  
0      PJ Nabarro           0   Patrick Nabarro  November 10, 2018  
1  Annalee Newitz           0           io9.com       May 23, 2018  
2    Sean Axmaker           0  Stream on Demand    January 4, 2018  
3   Daniel Kasman           0              MUBI  November 16, 2017  
4             NaN           0      Cinema Scope   October 12, 2017  


In [53]:
rt_reviews.isna().sum()

id                0
review         5563
rating        13517
fresh             0
critic         2722
top_critic        0
publisher       309
date              0
dtype: int64